# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

> **Người thực hiện:** Bùi Thị Thu Uyên · MSSV 2A202602613 · K4-Track02-Day18  
> **Đường chạy:** lightweight (`deltalake` 1.6.6, `pyiceberg` 0.12.0, DuckDB 1.5.6, Polars 1.44.2) · Python 3.11.7 · Windows 11  
> Notebook được thực thi từ đầu đến cuối; ô *Giải thích kết quả* ở cuối bài đọc các số in ở output phía trên.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 9de12248-c2c4-495d-aa86-f32c16… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 92193d7a-4c32-4efb-a9c0-88b3d5… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
# `ts` is a UTC timestamp. Without this, CAST(ts AS DATE) uses the machine's
# local time zone (Asia/Saigon here): the 7 UTC days become 8 local dates with
# partial first/last days, and Gold changes depending on who runs it.
con.sql("SET TimeZone = 'UTC'")
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 7,
 'numFilesRemoved': 7,
 'filesAdded': '{"avg":2769.0,"max":2770,"min":2766,"totalFiles":7,"totalSize":19383}',
 'filesRemoved': '{"avg":2688.1428571428573,"max":2689,"min":2687,"totalFiles":7,"totalSize":18817}',
 'partitionsOptimized': 7,
 'numBatches': 7,
 'totalConsideredFiles': 7,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table()).sort(["date", "model"])
with pl.Config(tbl_rows=30, tbl_width_chars=140, fmt_str_lengths=20):
    print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (21, 8)
┌────────────┬───────────────────┬────────────────┬────────────────┬─────────────────────┬───────────────────────┬────────────┬────────────┐
│ date       ┆ model             ┆ p50_latency_ms ┆ p95_latency_ms ┆ total_prompt_tokens ┆ total_completion_tok… ┆ error_rate ┆ cost_usd   │
│ ---        ┆ ---               ┆ ---            ┆ ---            ┆ ---                 ┆ ---                   ┆ ---        ┆ ---        │
│ date       ┆ str               ┆ f64            ┆ f64            ┆ decimal[38,0]       ┆ decimal[38,0]         ┆ f64        ┆ f64        │
╞════════════╪═══════════════════╪════════════════╪════════════════╪═════════════════════╪═══════════════════════╪════════════╪════════════╡
│ 2026-04-01 ┆ claude-haiku-4-5  ┆ 563.0          ┆ 1131.0         ┆ 16722403            ┆ 8258111               ┆ 0.048825   ┆ 46.410366  │
│ 2026-04-01 ┆ claude-opus-4-7   ┆ 3008.0         ┆ 6039.4         ┆ 5436642             ┆ 2709702               ┆ 0.05013    ┆ 284.77728  

## 🔎 Giải thích kết quả — NB4

**Bronze → Silver:** Bronze có **200,000** bản ghi thô (`raw_json` dạng chuỗi). Generator chèn **9,948** `request_id` trùng
để mô phỏng client retry. Silver parse JSON thành cột có kiểu, rồi dedup bằng
`ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts)` và giữ bản ghi sớm nhất, còn **190,052** dòng.
Con số này đúng bằng số `request_id` duy nhất, nên Silver < Bronze ✓. Nếu không dedup, các metric token, cost và error ở Gold
sẽ bị đếm thừa khoảng 5%.

**Gold (date × model):** **21 dòng = 7 ngày (2026-04-01 → 04-07) × 3 model** ✓. Các check trên mọi dòng đều đạt:
p50 ≤ p95, `cost_usd > 0`, `error_rate` nằm trong [0.045, 0.062] ⊂ [0, 1], không có null.
- p95 gần bằng 2× p50 ở cả 3 model (haiku 565/1,132 ms, sonnet 1,382/2,745 ms, opus 3,036/5,976 ms), tức latency có đuôi dài.
  Dashboard chỉ nhìn trung bình sẽ không thấy phần đuôi này.
- Chi phí 7 ngày (giá minh họa) là **$4,754.78**. Sonnet tốn nhiều nhất (**$2,419**) dù đơn giá thấp hơn opus, vì lượng token
  lớn hơn nhiều (~33M prompt token/ngày so với ~5.5M của opus). Opus có giá mỗi token cao nhất ($2,014) nhưng traffic ít hơn.

**Lỗi múi giờ mình phát hiện và sửa:** ở lần chạy đầu, Gold có **8 ngày / 24 dòng** dù generator sinh 7 ngày UTC.
Nguyên nhân là DuckDB `CAST(ts AS DATE)` dùng múi giờ máy (`Asia/Saigon`, UTC+7). Ngày 04-01 chỉ còn 20,239 dòng và
04-08 chỉ có 8,333 dòng, nên cost của ngày cuối bị "cụt" (~$80 so với ~$290 với opus). Check "≥ 7 ngày" vẫn PASS
nhưng số liệu sai, và kết quả thay đổi tùy múi giờ máy người chạy. Mình thêm `SET TimeZone = 'UTC'` thì Gold về đúng 7 ngày,
ngày nào cũng đủ dữ liệu. Bài học: Gold phải định nghĩa rõ ranh giới ngày, và một check về số lượng không phát hiện được lỗi
về ý nghĩa dữ liệu.

**Check bổ sung:** notebook gốc chưa assert hết yêu cầu Gold, nên mình thêm ô kiểm tra cho 3 tầng trên đĩa, Silver < Bronze,
≥ 7 ngày, 3 model, mỗi model có đủ mọi ngày, p50 ≤ p95, cost > 0, error_rate ∈ [0, 1] và không có null.

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

In [6]:
gold_stats = gold_df.select(
    pl.col("p50_latency_ms").min().alias("p50_min"),
    pl.col("p95_latency_ms").max().alias("p95_max"),
    pl.col("cost_usd").min().alias("cost_min"),
    pl.col("cost_usd").sum().alias("cost_total"),
    pl.col("error_rate").min().alias("err_min"),
    pl.col("error_rate").max().alias("err_max"),
).row(0, named=True)
print("Gold value ranges:", {k: round(v, 4) for k, v in gold_stats.items()})

per_model = (gold_df.group_by("model")
             .agg(pl.col("date").n_unique().alias("dates"),
                  pl.col("p50_latency_ms").mean().round(0).alias("avg_p50"),
                  pl.col("p95_latency_ms").mean().round(0).alias("avg_p95"),
                  pl.col("error_rate").mean().round(4).alias("avg_error_rate"),
                  pl.col("cost_usd").sum().round(2).alias("cost_usd_7d"))
             .sort("model"))
print(per_model)

checks = {
    "bronze/silver/gold on disk":      all((Path(p) / "_delta_log").exists() for p in (BRONZE, SILVER, GOLD)),
    "silver < bronze (dedup)":         silver_n < bronze_n,
    "gold ≥ 7 dates":                  n_dates >= 7,
    "gold has 3 models":               n_models == 3,
    "every model on every date":       gold_df.height == n_dates * n_models,
    "p50 ≤ p95 on every row":          gold_df.filter(pl.col("p50_latency_ms") > pl.col("p95_latency_ms")).height == 0,
    "cost_usd > 0 on every row":       gold_df.filter(pl.col("cost_usd") <= 0).height == 0,
    "error_rate in [0, 1]":            gold_df.filter(~pl.col("error_rate").is_between(0, 1)).height == 0,
    "no nulls in Gold metrics":        gold_df.null_count().sum_horizontal().item() == 0,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB4 incomplete — see FAIL rows above"
print("\nNB4 complete.")

Gold value ranges: {'p50_min': 553.0, 'p95_max': 6039.4, 'cost_min': 44.5939, 'cost_total': 4754.7848, 'err_min': 0.0451, 'err_max': 0.0615}
shape: (3, 6)
┌───────────────────┬───────┬─────────┬─────────┬────────────────┬─────────────┐
│ model             ┆ dates ┆ avg_p50 ┆ avg_p95 ┆ avg_error_rate ┆ cost_usd_7d │
│ ---               ┆ ---   ┆ ---     ┆ ---     ┆ ---            ┆ ---         │
│ str               ┆ u32   ┆ f64     ┆ f64     ┆ f64            ┆ f64         │
╞═══════════════════╪═══════╪═════════╪═════════╪════════════════╪═════════════╡
│ claude-haiku-4-5  ┆ 7     ┆ 565.0   ┆ 1132.0  ┆ 0.0489         ┆ 321.27      │
│ claude-opus-4-7   ┆ 7     ┆ 3036.0  ┆ 5976.0  ┆ 0.0522         ┆ 2014.34     │
│ claude-sonnet-4-6 ┆ 7     ┆ 1382.0  ┆ 2745.0  ┆ 0.0496         ┆ 2419.18     │
└───────────────────┴───────┴─────────┴─────────┴────────────────┴─────────────┘
  [PASS] bronze/silver/gold on disk
  [PASS] silver < bronze (dedup)
  [PASS] gold ≥ 7 dates
  [PASS] gold has 3 mod